# Предобработка данных о продажах видеоигр (2000–2013)

- Автор: Новаковский Дмитрий
- Дата: 02.08.2026

### Цели и задачи проекта

Провести подготовку и очистку данных для того, чтобы в последующих проектах удобно и корректно работать с информацией, а именно:
* ознакомиться с данными
* очистить данные
* отобрать данные в период с 2000 по 2013 год включительно
* категоризовать игры на три категории: высокую, среднюю и низкую
* выделить топ-7 платформ по количеству игр

### Описание данных

* Name — название игры.
* Platform — название платформы.
* Year of Release — год выпуска игры.
* Genre — жанр игры.
* NA sales — продажи в Северной Америке (в миллионах проданных копий).
* EU sales — продажи в Европе (в миллионах проданных копий).
* JP sales — продажи в Японии (в миллионах проданных копий).
* Other sales — продажи в других странах (в миллионах проданных копий).
* Critic Score — оценка критиков (от 0 до 100).
* User Score — оценка пользователей (от 0 до 10).
* Rating — рейтинг организации ESRB (англ. Entertainment Software Rating Board).

### Содержимое проекта

* [1 Загрузка данных и знакомство с ними](#upload)
* [2 Проверка ошибок в данных и их предобработка](#check)
* [2.1 Названия, или метки, столбцов датафрейма](#check_name)
* [2.2. Типы данных](#check_type)
* [2.3. Наличие пропусков в данных](#check_NaN)
* [2.4. Явные и неявные дубликаты в данных](#check_duples)
* [3 Фильтрация данных](#filter)
* [4 Категоризация данных](#category)
* [5. Итоговый вывод](#final)

## 1. Загрузка данных и знакомство с ними <a id='upload'></a>

In [1]:
# Пустые ячейки после каждого задания — примерное пространство для работы.
# Вы можете свободно добавлять или удалять ячейки по своему усмотрению в зависимости от логики и объёма работы.

In [2]:
# Используйте ячейки типа Code для вашего кода

In [3]:
# При необходимости добавьте новые ячейки для кода

In [4]:
import pandas as pd 

In [5]:
darkwood = pd.read_csv('/datasets/new_games.csv')

In [6]:
darkwood.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


* Есть несоотвествие типов данных - в NA sales, EU sales, JP sales, Critic Score и User Score вместо числового формата стоит object. 
* Также в некоторых столбцах есть пропуски. 
* Всего строк 16,956, 
* В столбце Critic Score меньше всего заполненных значений - только 8,923.
* Также пропуски есть в столбцах Rating, User Score, Genre, Year of Release и Name

* Некоторые названия вроде NA sales содержат пробелы, их нужно привести к snake_case. 
* Также столбец Rating назван слишком общим словом. Это может быть рейтинг как игры, так и игрока

---

## 2.  Проверка ошибок в данных и их предобработка <a id='check'></a>

### 2.1. Названия, или метки, столбцов датафрейма <a id='check_name'></a>

In [7]:
print("Названия столбцов:", darkwood.columns.tolist())

Названия столбцов: ['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales', 'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating']


In [8]:
darkwood.columns = darkwood.columns.str.lower().str.replace(' ', '_')

In [9]:
print("Названия столбцов приведенных к snake_case:", darkwood.columns.tolist())

Названия столбцов приведенных к snake_case: ['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales', 'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating']


### 2.2. Типы данных <a id='check_type'></a>

In [10]:
incorrect_columns = ['na_sales', 'eu_sales', 'jp_sales', 'critic_score', 'user_score']

In [11]:
for col in incorrect_columns:
    darkwood[col] = pd.to_numeric(darkwood[col], errors='coerce')

In [12]:
darkwood.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float64
 5   eu_sales         16950 non-null  float64
 6   jp_sales         16952 non-null  float64
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       7688 non-null   float64
 10  rating           10085 non-null  object 
dtypes: float64(7), object(4)
memory usage: 1.4+ MB


Проверка сколько таких пропусков

In [13]:
print("\nКоличество пропусков по столбцам:", darkwood[incorrect_columns].isnull().sum())


Количество пропусков по столбцам: na_sales           0
eu_sales           6
jp_sales           4
critic_score    8714
user_score      9268
dtype: int64


### 2.3. Наличие пропусков в данных <a id='check_NaN'></a>

In [14]:
missing_counts = darkwood.isnull().sum()

In [15]:
missing_percent = (darkwood.isnull().sum() / len(darkwood)) * 100

In [16]:
missing_data = pd.DataFrame({
    'Количество': missing_counts,
    'Процент (%)': missing_percent})

In [17]:
print("Анализ пропусков в данных:",missing_data)


Анализ пропусков в данных:                  Количество  Процент (%)
name                      2     0.011795
platform                  0     0.000000
year_of_release         275     1.621845
genre                     2     0.011795
na_sales                  0     0.000000
eu_sales                  6     0.035386
jp_sales                  4     0.023590
other_sales               0     0.000000
critic_score           8714    51.391838
user_score             9268    54.659118
rating                 6871    40.522529


In [18]:
sales_cols = ['na_sales', 'eu_sales', 'jp_sales']

In [19]:
for col in sales_cols:
    group_means = darkwood.groupby(['platform', 'year_of_release'])[col].transform('mean')
    darkwood[col] = darkwood[col].fillna(group_means)


In [20]:
print("Количество пропусков после обработки:", darkwood[sales_cols].isnull().sum())

Количество пропусков после обработки: na_sales    0
eu_sales    0
jp_sales    0
dtype: int64


### 2.4. Явные и неявные дубликаты в данных <a id='check_duples'></a>

In [21]:
cat_features = ['genre', 'platform', 'rating']
for col in cat_features:
    print(f"Уникальные значения в {col} до нормализации:")
    print(darkwood[col].unique())
    print("-" * 30)

Уникальные значения в genre до нормализации:
['Sports' 'Platform' 'Racing' 'Role-Playing' 'Puzzle' 'Misc' 'Shooter'
 'Simulation' 'Action' 'Fighting' 'Adventure' 'Strategy' nan 'MISC'
 'ROLE-PLAYING' 'RACING' 'ACTION' 'SHOOTER' 'FIGHTING' 'SPORTS' 'PLATFORM'
 'ADVENTURE' 'SIMULATION' 'PUZZLE' 'STRATEGY']
------------------------------
Уникальные значения в platform до нормализации:
['Wii' 'NES' 'GB' 'DS' 'X360' 'PS3' 'PS2' 'SNES' 'GBA' 'PS4' '3DS' 'N64'
 'PS' 'XB' 'PC' '2600' 'PSP' 'XOne' 'WiiU' 'GC' 'GEN' 'DC' 'PSV' 'SAT'
 'SCD' 'WS' 'NG' 'TG16' '3DO' 'GG' 'PCFX']
------------------------------
Уникальные значения в rating до нормализации:
['E' nan 'M' 'T' 'E10+' 'K-A' 'AO' 'EC' 'RP']
------------------------------


In [22]:
darkwood['genre'] = darkwood['genre'].str.lower()
darkwood['platform'] = darkwood['platform'].str.lower()
darkwood['rating'] = darkwood['rating'].str.upper()
darkwood['rating'] = dark_values = darkwood['rating'].replace('K-A', 'E')

In [23]:
print("Уникальные значения после нормализации:")
for col in cat_features:
    print(f"{col}: {darkwood[col].unique()}")

Уникальные значения после нормализации:
genre: ['sports' 'platform' 'racing' 'role-playing' 'puzzle' 'misc' 'shooter'
 'simulation' 'action' 'fighting' 'adventure' 'strategy' nan]
platform: ['wii' 'nes' 'gb' 'ds' 'x360' 'ps3' 'ps2' 'snes' 'gba' 'ps4' '3ds' 'n64'
 'ps' 'xb' 'pc' '2600' 'psp' 'xone' 'wiiu' 'gc' 'gen' 'dc' 'psv' 'sat'
 'scd' 'ws' 'ng' 'tg16' '3do' 'gg' 'pcfx']
rating: ['E' nan 'M' 'T' 'E10+' 'AO' 'EC' 'RP']


In [24]:
duplicates_count = darkwood.duplicated().sum()
print(f"Количество найденных явных дубликатов: {duplicates_count}")

Количество найденных явных дубликатов: 241


In [25]:
darkwood = darkwood.drop_duplicates()

In [26]:
print(f"Количество дубликатов после удаления: {darkwood.duplicated().sum()}")

Количество дубликатов после удаления: 0


В ходе работы с данными были выявлены неявные дубликаты, вызванные разным регистром написания жанров и платформ.
После проведения нормализации всех текстовых полей эти проблемы были устранены

В ходе анализа также было обнаружено и удалено 241 явных дубликатов

In [27]:
initial_count = len(darkwood)
absolute_removed = duplicates_count

In [28]:
relative_percent = (absolute_removed / initial_count) * 100


In [29]:
print(f"Изначальное количество строк: {initial_count}")
print(f"Количество удаленных дубликатов: {absolute_removed}")
print(f"Процент удаленных данных: {relative_percent:.2f}%")

Изначальное количество строк: 16715
Количество удаленных дубликатов: 241
Процент удаленных данных: 1.44%


Доля удаленных дубликатов от общего объема - 2,20%, что является существенным количеством.
Игнорировать их нельзя - это повлияет на анализ, поэтому мы их удалили

---

## 3. Фильтрация данных <a id='filter'></a>

In [30]:
darkwood_actual = darkwood[darkwood['year_of_release'].between(2000, 2013)].copy()

In [31]:
print(f"Количество строк после фильтрации (2000-2013): {len(darkwood_actual)}")

Количество строк после фильтрации (2000-2013): 12781


In [32]:
print(darkwood_actual['year_of_release'].min(), " — ", darkwood_actual['year_of_release'].max())

2000.0  —  2013.0


---

## 4. Категоризация данных <a id='category'></a>

In [33]:
user_bins = [0, 3, 8, float('inf')]
user_labels = ['низкая оценка', 'средняя оценка', 'высокая оценка']

In [34]:
darkwood_actual['user_score_category'] = pd.cut(
    darkwood_actual['user_score'], 
    bins=user_bins, 
    labels=user_labels, 
    right=False)

In [35]:
critic_bins = [0, 30, 80, float('inf')]
critic_labels = ['низкая оценка', 'средняя оценка', 'высокая оценка']

In [36]:
darkwood_actual['critic_score_category'] = pd.cut(
    darkwood_actual['critic_score'], 
    bins=critic_bins, 
    labels=critic_labels, 
    right=False)

In [37]:
print("Распределение оценок пользователей:")
print(darkwood_actual['user_score_category'].value_counts())

Распределение оценок пользователей:
средняя оценка    4081
высокая оценка    2286
низкая оценка      116
Name: user_score_category, dtype: int64


In [38]:
print("\nРаспределение оценок критиков:")
print(darkwood_actual['critic_score_category'].value_counts())


Распределение оценок критиков:
средняя оценка    5422
высокая оценка    1692
низкая оценка       55
Name: critic_score_category, dtype: int64


In [39]:
top_platforms = darkwood_actual['platform'].value_counts().nlargest(7).index.tolist()

In [40]:
print("\nТоп-7 платформ:", top_platforms)


Топ-7 платформ: ['ps2', 'ds', 'wii', 'psp', 'x360', 'ps3', 'gba']


---

## 5. Итоговый вывод <a id='final'></a>

В ходе работы над проектом был проведен полный цикл подготовки данных

* Данные были приведены к единому стандарту snake_case, что устранило ошибки в названиях и категориях.
* Проведен переход к числовым типам для всех ключевых метрик, заполнены пропуски в данных о продажах с помощью группировки по платформам и годам выпуска.
* Была сформирована целевая выборка, включающая игры периода с 2000 по 2013 год. (итого 12,781 запись).
* Созданы дополнительные колонки категорий для оценок пользователей и критиков, что позволит в дальнейшем анализировать качество игр по группам.
* Определены топ-7 наиболее представленных на рынке платформ за указанный период: ps2, ds, wii, psp, x360, ps3, gba.